In [16]:
import pandas as pd
from data import *

In [17]:
# Download data 
genes_to_disease = pd.read_csv("../data/genes_to_disease.txt", sep="\t")
genes_to_disease = genes_to_disease.drop(columns='source')

ppi = pd.read_csv("https://stringdb-downloads.org/download/stream/protein.links.v12.0/9606.protein.links.v12.0.min700.csv.gz", sep="," )

doc = pd.read_csv("https://stringdb-downloads.org/download/protein.info.v12.0/9606.protein.info.v12.0.txt.gz", sep="\t")
doc = doc.rename(columns={"preferred_name":"gene_symbol"})
doc = doc.drop(columns="protein_size")

profils_omim[["gene", "disease"]] = profils_omim["gene_maladie_assoc"].str.extract(r"(\w+)\s+\((\w+:\d+)\)")
profils_omim = profils_omim.drop(columns='gene_maladie_assoc')

In [18]:
# Jointure
df0 = pd.merge(genes_to_disease, doc, how='left', on='gene_symbol')
print(pd.isna(df0).value_counts())
df1 = pd.merge(df0, ppi, how='left', left_on="#string_protein_id", right_on="protein1")
print(pd.isna(df1).value_counts())
df1=df1.drop(columns="protein1")

ncbi_gene_id  gene_symbol  association_type  disease_id  #string_protein_id  annotation
False         False        False             False       False               False         15561
                                                         True                True            353
Name: count, dtype: int64
ncbi_gene_id  gene_symbol  association_type  disease_id  #string_protein_id  annotation  protein1  protein2  combined_score
False         False        False             False       False               False       False     False     False             727845
                                                         True                True        True      True      True                 353
                                                         False               False       True      True      True                 287
Name: count, dtype: int64


In [8]:
res = set(genes_to_disease['gene_symbol'])&set(doc['gene_symbol'])
print(len(res))
print(len(genes_to_disease['gene_symbol'].unique()))
print(len(set(ppi['protein1'])&set(doc[doc['gene_symbol'].isin(res)]['#string_protein_id'])))


5361
5505
5193


In [5]:
print(f"Protéines dans df0 et pas dans ppi : {len(set(df0["#string_protein_id"]))-len(set(df0["#string_protein_id"])&set(ppi["protein1"]))}")

Protéines dans df0 et pas dans ppi : 169


In [4]:
# High combined score
df1_HCS = df1[df1['combined_score']>=900]

df1_HCS = df1_HCS.groupby("#string_protein_id", as_index=False).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    disease_id=("disease_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list)
)
df1 = df1.groupby("#string_protein_id", as_index=False).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    disease_id=("disease_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list)
)
df1_HCS

,#string_protein_id,ncbi_gene_id,gene_symbol,association_type,disease_id,annotation,protein2
0,9606.ENSP00000001146,NCBIGene:56603,CYP26B1,MENDELIAN,OMIM:614416,Cytochrome P450 26B1; Involved in the metaboli...,"[9606.ENSP00000260682, 9606.ENSP00000360958, 9..."
1,9606.ENSP00000003084,NCBIGene:1080,CFTR,POLYGENIC,OMIM:167800,Cystic fibrosis transmembrane conductance regu...,"[9606.ENSP00000366488, 9606.ENSP00000364235, 9..."
2,9606.ENSP00000003100,NCBIGene:1595,CYP51A1,UNKNOWN,ORPHA:521432,Lanosterol 14-alpha demethylase; A cytochrome ...,"[9606.ENSP00000359297, 9606.ENSP00000360316, 9..."
3,9606.ENSP00000005226,NCBIGene:10083,USH1C,MENDELIAN,OMIM:276900,Harmonin; USH1 protein network component harmo...,"[9606.ENSP00000384582, 9606.ENSP00000386331, 9..."
4,9606.ENSP00000005257,NCBIGene:5898,RALA,MENDELIAN,OMIM:619311,Ras-related protein Ral-A; Multifunctional GTP...,"[9606.ENSP00000394560, 9606.ENSP00000484855, 9..."
...,...,...,...,...,...,...,...
4605,9606.ENSP00000500986,NCBIGene:80821,DDHD1,MENDELIAN,OMIM:609340,Phospholipase DDHD1; Phospholipase that hydrol...,"[9606.ENSP00000380352, 9606.ENSP00000380352]"
4606,9606.ENSP00000500990,NCBIGene:6622,SNCA,MENDELIAN,OMIM:605543,Alpha-synuclein; Neuronal protein that plays s...,"[9606.ENSP00000419081, 9606.ENSP00000496339, 9..."
4607,9606.ENSP00000501092,NCBIGene:7020,TFAP2A,MENDELIAN,OMIM:113620,Transcription factor AP-2-alpha; Sequence-spec...,"[9606.ENSP00000478887, 9606.ENSP00000377265, 9..."
4608,9606.ENSP00000501111,NCBIGene:1056,CEL,MENDELIAN,OMIM:609812,Bile salt-activated lipase; Catalyzes the hydr...,"[9606.ENSP00000011292, 9606.ENSP00000358232, 9..."


In [5]:
print(len(set(df1['disease_id'])&set(df_hpoa['database_id'])))
print(len(df_hpoa['database_id'].unique()))
print(len(df1['disease_id'].unique()))
print(len(set(genes_to_disease['disease_id'].unique())&set(df_hpoa['database_id'].unique())))

4688
12996
4928
9120


In [9]:
print("========= Maladies en commun avec df1 ==========")
print("Dataset : profils_omim")
print(f"{len(set(profils_omim['disease'].unique())&set(df1['disease_id'].unique()))}/{len(profils_omim['disease'].unique())}")
print("Dataset : df_omim")
print(f"{len(set(df_omim['database_id'].unique())&set(df1['disease_id'].unique()))}/{len(df_omim['database_id'].unique())}")
print("Dataset : df_orpha")
print(f"{len(set(df_orpha['database_id'].unique())&set(df1['disease_id'].unique()))}/{len(df_orpha['database_id'].unique())}")


========= Maladies en commun avec df1 ==========
Dataset : profils_omim
4158/6139
Dataset : df_omim
270/550
Dataset : df_orpha
14/550


In [10]:
print(len(set(df_omim['database_id'].unique())&set(genes_to_disease['disease_id'].unique())))
print(len(set(df_orpha['database_id'].unique())&set(genes_to_disease['disease_id'].unique())))

418
397


In [11]:
genes_correspondence={}
for i in range(correspondence_exacte.shape[0]):
    omim = correspondence_exacte["omim_id"][i]
    orpha = correspondence_exacte["orpha_id"][i]
    if omim in genes_to_disease["disease_id"].values:
        gene_omim=genes_to_disease[genes_to_disease['disease_id']==omim]['gene_symbol']
    else:
        gene_omim=None
    if orpha in genes_to_disease["disease_id"].values:
        gene_orpha=genes_to_disease[genes_to_disease['disease_id']==orpha]['gene_symbol']
    else:
        gene_orpha=None
    genes_correspondence[(omim, orpha)]=(gene_omim, gene_orpha)

print("Nombre de maladies correspondantes qui n'ont pas exactement les mêmes gènes :", 
sum(
    v[0] is not None and v[1] is not None and set(v[0]) != set(v[1])
    for v in genes_correspondence.values()
))
print([k for k,v in genes_correspondence.items() if v[0] is not None and v[1] is not None and set(v[0]) != set(v[1])])
print("Nombre de paires qui ont au moins un None :", len(genes_correspondence)-
sum(v[0] is not None and v[1] is not None for v in genes_correspondence.values()))

Nombre de maladies correspondantes qui n'ont pas exactement les mêmes gènes : 60
[('OMIM:268000', 'ORPHA:791'), ('OMIM:263800', 'ORPHA:358'), ('OMIM:601321', 'ORPHA:638'), ('OMIM:219000', 'ORPHA:2052'), ('OMIM:615237', 'ORPHA:2301'), ('OMIM:612376', 'ORPHA:520'), ('OMIM:242600', 'ORPHA:42062'), ('OMIM:167400', 'ORPHA:46348'), ('OMIM:269250', 'ORPHA:3144'), ('OMIM:218330', 'ORPHA:1515'), ('OMIM:601859', 'ORPHA:3261'), ('OMIM:273800', 'ORPHA:849'), ('OMIM:225500', 'ORPHA:289'), ('OMIM:254500', 'ORPHA:29073'), ('OMIM:143100', 'ORPHA:399'), ('OMIM:277590', 'ORPHA:3447'), ('OMIM:603554', 'ORPHA:39041'), ('OMIM:236730', 'ORPHA:2704'), ('OMIM:123150', 'ORPHA:1540'), ('OMIM:117550', 'ORPHA:821'), ('OMIM:229200', 'ORPHA:90354'), ('OMIM:606764', 'ORPHA:44890'), ('OMIM:208150', 'ORPHA:994'), ('OMIM:214800', 'ORPHA:138'), ('OMIM:254940', 'ORPHA:1358'), ('OMIM:600880', 'ORPHA:131'), ('OMIM:608572', 'ORPHA:1200'), ('OMIM:102370', 'ORPHA:969'), ('OMIM:219700', 'ORPHA:586'), ('OMIM:216340', 'ORPHA:347

In [10]:
print(f"Nombre de maladies ORPHA : {len(genes_to_disease[genes_to_disease['disease_id'].str.startswith("ORPHA")])}")
print(f"Nombre de maladies OMIM : {len(genes_to_disease[genes_to_disease['disease_id'].str.startswith("OMIM")])}")

def find_gene_correspondence(df, disease_col, gene_col):
    gene_sets = df.groupby(disease_col)[gene_col].apply(set)
    omim_sets = gene_sets[gene_sets.index.str.startswith("OMIM")]
    orpha_sets = gene_sets[gene_sets.index.str.startswith("ORPHA")]
    matches = []
    for omim_id, omim_genes in omim_sets.items():
        for orpha_id, orpha_genes in orpha_sets.items():
            if omim_genes == orpha_genes:
                matches.append({"omim_id": omim_id, "orpha_id": orpha_id, "genes": ", ".join(omim_genes)})
    
    return pd.DataFrame(matches)
    
df_gene_correspondence=find_gene_correspondence(genes_to_disease, 'disease_id', 'gene_symbol')


Nombre de maladies ORPHA : 8290
Nombre de maladies OMIM : 7624


In [11]:
print(df_gene_correspondence.shape)
print(len(set(correspondence_exacte['omim_id'].unique())&set(df_gene_correspondence['omim_id'].unique())))
print(len(set(df_gene_correspondence['orpha_id'].unique())&set(df_hpoa['database_id'].unique())))
df_gene_correspondence0=df_gene_correspondence.loc[df_gene_correspondence['orpha_id'].isin(df_hpoa['database_id'])]
print(df_gene_correspondence0.shape)

(6375, 3)
365
1700
(3858, 3)


In [12]:
list_omim = df_gene_correspondence0['omim_id'].unique()
list_orpha = df_gene_correspondence0['orpha_id'].unique()
len(list_omim), len(list_orpha)

work_omim = df_pivot[df_pivot['database_id'].isin(list_omim)]
work_orpha = df_pivot[df_pivot['database_id'].isin(list_orpha)]

work_omim.shape, work_orpha.shape

((2247, 11614), (1700, 11614))

In [14]:
df1_orpha = pd.merge(work_orpha, df1, how='left', left_on='database_id', right_on='disease_id')
# print(df1_orpha.isna().sum())
print(df1_orpha.shape)
df1_omim = pd.merge(work_omim, df1, how='left', left_on='database_id', right_on='disease_id')
print(df1_omim.shape) #, df1_omim.isna().sum())

(79878, 11622)
(108575, 11622)


In [22]:
df1_omim

,database_id,HP:0000002,HP:0000003,HP:0000006,HP:0000007,HP:0000008,HP:0000009,HP:0000010,HP:0000011,HP:0000012,...,HP:6001440,HP:6001454,ncbi_gene_id,gene_symbol,association_type,disease_id,#string_protein_id,annotation,protein2,combined_score
0,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000493985,903.0
1,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000078429,908.0
2,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000262958,717.0
3,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000319713,715.0
4,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000360021,910.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108570,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000307567,784.0
108571,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000360483,766.0
108572,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000355889,895.0
108573,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000199389,802.0


In [ ]:
df1_omim = df1_omim.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)

KeyError: "Label(s) ['#string_protein_id'] do not exist"

In [26]:
df1_omim.isna().sum()

disease_id           0
ncbi_gene_id         0
gene_symbol          0
association_type     0
protein             28
annotation          28
protein2             0
combined_score       0
dtype: int64

In [ ]:
df1_orpha = df1_orpha.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)
df1_orpha.shape

(1700, 8)

In [28]:
df1_orpha.isna().sum()

disease_id           0
ncbi_gene_id         0
gene_symbol          0
association_type     0
protein             21
annotation          21
protein2             0
combined_score       0
dtype: int64